# Keyword Baseline
- [X] Build keyword rules for all 10 categories.
- [X] Document rule sources and rationale.
- [ ] Generate matches.
- [ ] Freeze rules.
- [ ] Select thresholds.
- [ ] Run validation.
- [ ] Have another teammate review the rules/results.

# Rule Architecture

In [1]:
from enum import Enum
from dataclasses import dataclass

class EvidenceEffect(Enum):
    """
    Fixed evidence effect on keyword baseline score
    """
    ANCHOR = 2,
    SUPPORT = 1,
    COUNTER = -1,
    VETO = 0

@dataclass
class Rule:
    """
    A single rule within an evidence group
    Attributes:
        rule_id: Unique identifier for the rule.
        phrases: Phrases that trigger this rule.
        source: Allowed source from which the rule was derived.
        rationale: Reason the rule was accepted and how it relates to the category.
        reviewer: Person who reviewed or approved the rule, when applicable.
        behavior: Intended behavior of the rule, such as inclusion or exclusion.
    """
    rule_id: str
    phrases: list[str]
    source: str
    rationale: str
    reviewer: str
    behavior: str


@dataclass
class EvidenceGroup:
    """
    A group of related keyword rules that contributes to a category score.
    Rules within a group represent synonymous or overlapping evidence and contribute at most once per chunk.
    Attributes:
        group_id: Unique identifier for the evidence group.
        effect: Type and score contribution of the group when matched.
        rules: Keyword rules belonging to this group.
    """
    group_id: str
    effect: EvidenceEffect
    rules: list[Rule]


@dataclass
class Scorecard:
    """
    Represents a scorecard, one for each of the final 10 categories
    """
    category: str
    anchor_groups: list[EvidenceGroup]
    supporting_groups: list[EvidenceGroup]
    counterevidence_groups: list[EvidenceGroup]
    veto_groups: list[EvidenceGroup]

# Defining Scorecards

In [2]:
from utils import load_file_from_github
category_evidence = load_file_from_github("notebooks/task3_outputs/category_evidence.json")

print("Final 10 Categories:")
for category in category_evidence.keys():
    print(category)

Final 10 Categories:
Governing Law
Renewal Term
Revenue/Profit Sharing
Cap On Liability
Uncapped Liability
Termination For Convenience
Anti-Assignment
Audit Rights
License Grant
Exclusivity


In [3]:
def print_positive_examples(category):
    """ Prints positive examples for a given category from evidence (Task 7). """
    for i, example in enumerate(category_evidence[category]["positive_examples"], start=1):
        print(f"{i}. {example["answer_text"]}\n")

In [4]:
def print_negative_examples(category):
    """ Prints negative examples for a given category from evidence (Task 7). """
    for i, example in enumerate(category_evidence[category]["hard_negative_examples"], start=1):
        print(f"{i}. {example["answer_text"]}\n")

## Governing Law
Description: Which state/country's law governs the interpretation of the contract?

In [5]:
print_positive_examples("Governing Law")

1. This Agreement shall be governed by and construed in accordance with the laws of the State of California without regard to its conflict of laws provisions.

2. This Agreement shall be governed by and construed in accordance with the laws of the Province of Ontario and the federal laws of Canada applicable in the Province of Ontario.

3. This Amendment shall be governed by and construed in accordance with the laws of Japan.

4. This Agreement shall be governed by and construed in accordance with the laws of New York, US, without reference to its conflict of laws principles, and shall not be governed by the United Nations Convention of International Contracts on the Sale of Goods (the Vienna Convention).

5. This Agreement will be governed by, construed and enforced in accordance with the laws of the State of Texas.



In [6]:
print_negative_examples("Governing Law")

1. Consequently either this Agreement or any of the respective rights or obligations of the Parties hereunder may be assigned or otherwise transferred, in whole or in part, in any form whatsoever (including by way of change of Control), by either Party subject to the prior written consent of the other Party, which consent shall not be unreasonably withheld or delayed, and any attempt to do so without such consent shall be null and void.

2. Either Party may, without consent of the other Party, assign this Agreement in whole to (i) in the case of RevMed, its successor in interest or assignee or purchaser, as applicable, in the case of a Change of Control or (ii) in the case of Sanofi, its successor in interest or assignee or purchaser, as applicable, in connection with the sale of all or substantially all of its assets to which this Agreement relates, or in connection with a merger, acquisition or similar transaction.

3. Neither this Agreement nor any rights or obligations hereunder sh

In [7]:
governing_law_scorecard = Scorecard(
    category="Governing Law",
    anchor_groups=[
        EvidenceGroup(
            group_id="GL-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="GL-R-001",
                    phrases=[
                        "shall be governed by and construed in accordance with the laws",
                    ],
                    source="annotated training Span",
                    rationale="Directly identifies the laws governing the agreement.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
                Rule(
                    rule_id="GL-R-002",
                    phrases=[
                        "will be governed by, construed and enforced in accordance with the laws",
                    ],
                    source="annotated training Span",
                    rationale="Directly identifies the governing law using a drafting variant that also specifies construction and enforcement.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[],
    counterevidence_groups=[],
    veto_groups=[],
)

The positive examples consistently identify the law governing the agreement, including formulations addressing governing, construction, or enforcement. The negative examples concern assignment, change of control, and license rights and do not establish governing-law evidence. The scorecard therefore relies on explicit governing-law formulations rather than standalone jurisdiction names, which could occur elsewhere in a contract without identifying the governing law.

## Renewal Term
Description: What is the renewal term after the initial term expires? This includes automatic extensions and unilateral extensions with prior notice.

In [8]:
print_positive_examples("Renewal Term")

1. Within ninety (90) days of our receipt of your notice to renew, we will furnish you with written notice of:  (i) reasons which could cause us not to grant a renewal to you including but not limited to any deficiencies which require correction and a schedule for correction by you; and (ii) our then-current requirements relating to the image, appearance, decoration, furnishing, equipping and stocking of Buffalo Wild Wings businesses, and a schedule for effecting upgrading or modifications in order to bring the Franchised Restaurant in compliance, as a condition of renewal.  Renewal of the franchise shall be conditioned upon your compliance with such requirements and continued compliance with all the terms and conditions of this Agreement up to the date of termination of the initial term.

2. This Agreement will automatically be renewed for periods of       twelve (12) months unless either Party gives six (6) months written       notice of its intent to terminate this Agreement.

3. Th

In [9]:
print_negative_examples("Renewal Term")

1. Either party may terminate this Agreement at any time with 30 days written notice.

2. ISO may terminate this Agreement prior to its expiration for cause upon prior written notice to SERVICERS as follows:

3. EHS or EHN, on the one hand, and Dr. Murray, on the other, may terminate any Services Term of this Agreement by delivering 60 days written notice to the other party.

4. ACTION FIRST ARISING, EVEN IF IT IS A CONTINUOUS ONE, OR IN THE          AGGREGATE, WITH RESPECT OF ALL CLAIMS ARISING OUT OF OR RELATED TO          THIS AGREEMENT, THE TOTAL AMOUNT ACTUALLY PAID BY CORIO UNDER THIS          AGREEMENT TO CHANGEPOINT.

5. Licensee shall have the non-exclusive right and license to develop and distribute ICE AGE 2 Wireless Products during the Term of the Agreement for all Wireless Products set forth in this Paragraph 2(c) except the Game, for which Licensee shall have the exclusive right and license to develop and distribute until December 31, 2006.



In [10]:
renewal_term_scorecard = Scorecard(
    category="Renewal Term",
    anchor_groups=[
        EvidenceGroup(
            group_id="RT-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="RT-R-001",
                    phrases=[
                        "automatically be renewed for periods",
                        "renewed for additional periods",
                    ],
                    source="annotated training Span",
                    rationale="Directly identifies additional periods during which the agreement is renewed.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
                Rule(
                    rule_id="RT-R-002",
                    phrases=[
                        "automatic successive renewal terms",
                    ],
                    source="annotated training Span",
                    rationale="Directly identifies successive periods designated as renewal terms.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="RT-A-002",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="RT-R-003",
                    phrases=[
                        "automatic extension for consecutive",
                    ],
                    source="annotated training Span",
                    rationale="Identifies recurring extension periods following the initial term.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="RT-A-003",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="RT-R-004",
                    phrases=[
                        "each such additional period, a \"Renewal Term\"",
                        "the \"Renewal Terms\"",
                    ],
                    source="annotated training Span",
                    rationale="Explicitly designates an additional contractual period as a Renewal Term.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[],
    counterevidence_groups=[],
    veto_groups=[],
)

The positive examples consistently establish renewal or extension periods following the initial term, including automatic renewal, successive renewal terms, and explicitly defined Renewal Terms. The negative examples include general termination provisions, termination for cause, service-term termination, liability provisions, and license grants, none of which establish a renewal period. The scorecard therefore focuses on language identifying additional, successive, or explicitly designated renewal periods rather than generic references to notice or termination.

## Revenue/Profit Sharing
Description: Is one party required to share revenue or profit with the counterparty for any technology, goods, or services?

In [11]:
print_positive_examples("Revenue/Profit Sharing")

1. In consideration for the intangible rights granted hereunder, for each Year in which the Spoken-Word Audio Sub-Section (including the Mirror Company Site) generates revenue of at [***] (the "Revenue Threshold"), Company will pay ACSI a royalty equal to [***] of all revenues generated from the Spoken-Word Audio Sub-Section (including, for the avoidance of doubt, any revenue received by Company from any Company customer who first links to the Mirror Company Site from the Spoken-Word Audio Sub-Section and who later accesses the Company Site directly) in excess of Revenue Threshold (the "Royalties") for each Year of the Term.

2. HSNS agrees to pay E.piphany an additional                   $0.005 per email for any email distributed by HSNS as a result                   of any deal it closes that either results from a lead                   generated by E.piphany or in which E.piphany assisted prior to                   closing for the first year after the deal closes.

3. During the Dis

In [12]:
print_negative_examples("Revenue/Profit Sharing")

1. This right to audit shall remain in effect throughout the life of this Agreement and for a period of three (3) years after the termination of this Agreement.

2. We have the right to inspect the proposed supplier's facilities, and require that product samples from the proposed supplier be delivered, at our option, either directly to us, or to any independent, certified laboratory that we may designate, for testing.

3. The auditor will only examine such books and records during business hours but not more than once each fiscal year while this Agreement remains in effect and for three years thereafter in order to verify expenses, Net Sales, Depomed Net Sales, PDEs or Details completed, or payments due under this Agreement.

4. At the end of such initial term, and any renewed term, as applicable, this Agreement shall automatically renew for an additional one (1) year term, unless a party provides written notice to the other parties at least six (6) months prior to the end of the

5. S

In [13]:
revenue_profit_sharing_scorecard = Scorecard(
    category="Revenue/Profit Sharing",
    anchor_groups=[
        EvidenceGroup(
            group_id="RPS-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="RPS-R-001",
                    phrases=[
                        "will pay ACSI a royalty equal to",
                        "royalty payment",
                        "cash royalty based on a percentage",
                    ],
                    source="annotated training Span",
                    rationale="Directly identifies a royalty payment calculated from revenue, profit, or another financial measure.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="RPS-A-002",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="RPS-R-002",
                    phrases=[
                        "additional $0.005 per email",
                        "pay ... a transfer price per unit of Product supplied",
                    ],
                    source="annotated training Span",
                    rationale="Identifies a payment obligation calculated on the volume or transaction activity associated with the underlying commercial arrangement.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[],
    counterevidence_groups=[],
    veto_groups=[],
)

The positive examples establish payments calculated by reference to revenue, sales, royalties, invoice value, shipping profit, or transaction volume. However, the examples do not establish that every transaction-based or per-unit payment is necessarily revenue or profit sharing. The negative examples concern audit, inspection, renewal, and franchise provisions and do not establish a competing financial-sharing pattern. The scorecard should therefore rely on payment language that is explicitly tied to royalties, revenue, sales, profit, or another identified financial measure, rather than treating generic prices, fees, or per-unit payments as sufficient evidence.

## Cap On Liability
Description: Does the contract include a cap on liability upon the breach of a party’s obligation? This includes time limitation for the counterparty to bring claims or maximum amount for recovery.

In [14]:
print_positive_examples("Cap On Liability")

1. If a Buyer receives a product that fails to conform to these representations and warranties, the sole remedies of Buyer for the breach of warranty will be to: (1) reject and return the non-conforming product to Seller for a refund or credit, or a replacement conforming product, in the manner and time period provided in the SOP; (2) obtain reimbursement from Seller for actual, reasonable, substantiated out-of-pocket expenses incurred by Buyer in the recovery, return or disposal of a non-conforming product that is the subject of a mandatory product recall required under Applicable Laws or a voluntary withdrawal declared by Seller or approved by Seller (such approval not to be unreasonably withheld, conditioned or delayed); and (3) obtain indemnification from Seller for any Indemnified Claim arising from or related to the non-conforming product as provided in Section 7.

2. In addition, in the event of a material breach by Nexstar of its obligations hereunder, WYZZ shall be entitled to

In [15]:
print_negative_examples("Cap On Liability")

1. The provisions of Section 9.1 and Section 9.2 will not apply to limit the Licensee's indemnification obligations under Section 8.2, or in the case of Licensee's gross negligence or wilful misconduct.

2. THE LIMITATIONS ON LIABILITY FOR DAMAGES SET FORTH IN THIS AGREEMENT SHALL BE INAPPLICABLE TO EACH PARTY'S CONTRACTUAL OBLIGATION TO INDEMNIFY THE OTHER PARTY AS SET FORTH IN SECTIONS 2.6 AND 13.

3. TO THE MAXIMUM EXTENT PERMISSIBLE UNDER APPLICABLE LAW, EXCEPT FOR THE WILFUL MISAPPROPRIATION OR INFRINGEMENT OF THE INTELLECTUAL PROPERTY OF A PARTY TO THIS AGREEMENT, OR THE OBLIGATIONS OF THE PARTIES TO THIS AGREEMENT PURSUANT TO SECTION 13, (A) THE LIABILITY OF ANY PARTY TO THIS AGREEMENT, IF ANY, FOR DAMAGES FOR ANY CLAIM OF ANY KIND WHATSOEVER AND REGARDLESS OF THE LEGAL THEORY, WITH REGARD TO THE RIGHTS GRANTED HEREUNDER OR THE SERVICES PERFORMED HEREUNDER, SHALL NOT INCLUDE COMPENSATION, REIMBURSEMENT OR DAMAGES ON ACCOUNT OF THE LOSS OF PRESENT OR PROSPECTIVE PROFITS, EXPENDIT

In [16]:
cap_on_liability_scorecard = Scorecard(
    category="Cap On Liability",
    anchor_groups=[
        EvidenceGroup(
            group_id="COL-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="COL-R-001",
                    phrases=[
                        "may not assert consequential, special or punitive damages",
                        "in no event will either party be liable for any special, indirect, incidental or consequential damages",
                    ],
                    source="annotated training Span",
                    rationale="Directly limits the categories of damages for which a party may be liable.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="COL-A-002",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="COL-R-002",
                    phrases=[
                        "up to a maximum of twelve (12) months of applicable charges",
                    ],
                    source="annotated training Span",
                    rationale="Establishes an express maximum on the amount of contractual credit available.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[],
    counterevidence_groups=[
        EvidenceGroup(
            group_id="COL-C-001",
            effect=EvidenceEffect.COUNTER,
            rules=[
                Rule(
                    rule_id="COL-R-003",
                    phrases=[
                        "will not apply to limit",
                        "limitations on liability for damages ... shall be inapplicable",
                    ],
                    source="annotated training Span",
                    rationale="Expressly states that an otherwise applicable liability limitation does not apply to specified obligations.",
                    reviewer="<reviewer>",
                    behavior="exclude",
                ),
            ],
        ),
    ],
    veto_groups=[],
)

The positive examples establish contractual limits on damages or monetary exposure, including exclusions of specified damages and express maximum amounts. The negative examples also show an important distinction: language stating that a liability limitation does not apply to specified obligations represents a carveout rather than the cap itself. The scorecard therefore distinguishes an underlying limitation from language identifying obligations that fall outside that limitation.

## Uncapped Liability
Description: Is a party’s liability uncapped upon the breach of its obligation in the contract? This also includes uncap liability for a particular type of breach such as IP infringement or breach of confidentiality obligation.

In [17]:
print_positive_examples("Uncapped Liability")

1. Subject to Section 17(c), in no event shall either Party's liability under this Agreement exceed the aggregate of all amounts paid under this Agreement and amounts that have accrued but not yet been paid in the twelve (12) months preceding the event giving rise to the claim.

2. In no event will either party be liable to the other for special, incidental, or indirect damages or for any consequential damages (including lost profits or savings), even if they are informed of the possibility; provided that this Section 10.0 does not apply to Customer's failure to pay any amounts owing to IBM under this Agreement (including amounts owing for Services that would have been rendered but for Customer's breach of this Agreement).

3. With the exception of wilful misconduct by a Party, and such cases where a limitation of liability and/or indemnification is not possible under applicable law, for which cases there shall be no limitation, any and all liability and/or indemnification obligations 

In [18]:
print_negative_examples("Uncapped Liability")

1. THE PROVIDER SHALL NOT BE LIABLE FOR ANY INCIDENTAL OR CONSEQUENTIAL DAMAGES, INCLUDING BUT NOT LIMITED TO LOSS OF REVENUES OR LOSS OF PROFIT, ARISING OUT OF THE PROVISION OF THE SERVICES OR REPAIRS, NOTWITHSTANDING ADVANCE NOTICE FROM THE COMPANY THAT THE POSSIBILITY OF SUCH DAMAGE OR LOSS EXISTS.

2. This includes in particular the liability exclusion for lost profit, the loss of data or interruption to or errors in the operation of the Web site of the Affiliate.

3. It is expressly agreed that the Agent shall not be liable for any loss, liability, claim, damage or expense or be  required to contribute any amount pursuant to Section 9(b) or this Section 10 which in the aggregate exceeds the amount paid (excluding  reimbursable expenses) to the Agent under this Agreement.

4. This Agreement will be construed and governed in accordance with the laws of the State of Illinois, without regard to conflict of laws principles.

5. WGT reserves all rights in and to the Trademarks and all  

In [19]:
uncapped_liability_scorecard = Scorecard(
    category="Uncapped Liability",
    anchor_groups=[
        EvidenceGroup(
            group_id="UL-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="UL-R-001",
                    phrases=[
                        "for which cases there shall be no limitation",
                    ],
                    source="annotated training Span",
                    rationale="Expressly states that liability is not subject to a limitation in specified circumstances.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="UL-A-002",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="UL-R-002",
                    phrases=[
                        "does not apply to Customer's failure to pay any amounts owing",
                    ],
                    source="annotated training Span",
                    rationale="Expressly identifies a payment obligation as outside an otherwise applicable damages limitation.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
                Rule(
                    rule_id="UL-R-003",
                    phrases=[
                        "except in connection with a breach by either party",
                    ],
                    source="annotated training Span",
                    rationale="Identifies specified breach-related obligations as exceptions to an otherwise applicable damages limitation.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
                Rule(
                    rule_id="UL-R-004",
                    phrases=[
                        "except for liability arising from section 9.3",
                    ],
                    source="annotated training Span",
                    rationale="Identifies specified liability as an exception to an otherwise applicable monetary limitation.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[],
    counterevidence_groups=[
        EvidenceGroup(
            group_id="UL-C-001",
            effect=EvidenceEffect.COUNTER,
            rules=[
                Rule(
                    rule_id="UL-R-005",
                    phrases=[
                        "shall not be liable for any incidental or consequential damages",
                        "liability exclusion for lost profit",
                    ],
                    source="annotated training Span",
                    rationale="Describes an ordinary exclusion of specified damages without identifying liability that remains outside an applicable limitation.",
                    reviewer="<reviewer>",
                    behavior="exclude",
                ),
            ],
        ),
    ],
    veto_groups=[],
)

The positive examples primarily establish exceptions to an otherwise applicable liability limitation, including specified breaches, payment obligations, indemnification obligations, or circumstances in which no limitation applies. The negative examples include ordinary damages exclusions and express monetary caps, which do not by themselves establish uncapped liability. The scorecard therefore focuses on language identifying a specific carveout from an otherwise applicable limitation, rather than treating general statements that a party is not liable or that certain damages are excluded as uncapped liability.

## Termination For Convenience
Description: Can a party terminate this contract without cause (solely by giving a notice and allowing a waiting  period to expire)?

In [20]:
print_positive_examples("Termination For Convenience")

1. Written notice of intention to withdraw must be served in writing upon the remaining Participants at least Thirty (30) business days prior to the withdrawal date.

2. The Agreement rests, for all that, cancellable at any time by any of the parties before the expiry date of the Agreement or any of itsrenewals, upon three months prior written notice.

3. Notwithstanding the foregoing, either party may terminate this Agreement at any time without liability by providing one hundred eighty (180) days written notice to the other party.

4. Party A is entitled to unilaterally terminate this Agreement within three natural months from the signing date of this Agreement.

5. Licensee may terminate this Agreement for convenience upon eighteen (18) months' advance written notice to Bioeq; provided, however, that any such termination for convenience shall not become effective prior to twelve (12) months after the First Commercial Sale of the first Licensed Product.



In [21]:
print_negative_examples("Termination For Convenience")

1. MBE shall have the right to elect by written notice to the Company at any time between two (2) and six (6) months prior to the end of such initial term or any subsequent Renewal Period (as defined below), to notify the Company that MBE elects to seek to extend such term for additional two (2)-year periods (each a "Renewal Period") In the event of such election, MBE and the Company shall have  -------------- a period of sixty (60) days in which to negotiate commercially reasonable Basic Fees, Bounty Fees and eBay Fees (and other applicable fees) ("Fee Schedule")                                                               ------------ under which the Company would be willing to renew this Agreement for such Renewal Period.

2. Upon expiry of the Initial Term, this Agreement [*****] unless a Notice of non-renewal is given by either Party to the other Party [*****] prior to the expiry of the Initial Term or the end of a renewal period, if any.

3. This Agreement shall commence on the 

In [22]:
termination_for_convenience_scorecard = Scorecard(
    category="Termination for Convenience",
    anchor_groups=[
        EvidenceGroup(
            group_id="TFC-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="TFC-R-001",
                    phrases=[
                        "cancellable at any time by any of the parties",
                        "terminate this Agreement at any time without liability by providing",
                    ],
                    source="annotated training Span",
                    rationale="Expressly permits termination or cancellation at the discretion of a party without requiring breach or another specified cause.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="TFC-A-002",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="TFC-R-002",
                    phrases=[
                        "terminate this Agreement for convenience upon",
                    ],
                    source="annotated training Span",
                    rationale="Directly identifies termination for convenience and specifies the required advance notice.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="TFC-A-003",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="TFC-R-003",
                    phrases=[
                        "unilaterally terminate this Agreement within",
                    ],
                    source="annotated training Span",
                    rationale="Identifies unilateral termination by a party without stating breach, default, or another cause as the basis.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="TFC-A-004",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="TFC-R-004",
                    phrases=[
                        "written notice of intention to withdraw",
                    ],
                    source="annotated training Span",
                    rationale="Identifies an express withdrawal mechanism initiated through advance written notice rather than a breach-based termination condition.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[],
    counterevidence_groups=[],
    veto_groups=[],
)

The positive examples consistently establish termination or withdrawal initiated by a party without requiring breach, default, or another specified cause, including termination at any time, unilateral termination, and termination expressly described as being for convenience. The negative examples primarily concern renewal rights, non-renewal, the general contractual term, exclusivity, and claim limitations. The scorecard therefore relies on discretionary termination or withdrawal language and does not treat generic terms such as “terminate,” “notice,” or a notice period alone as sufficient evidence.

## Anti-Assignment
Description: Is consent or notice required of a party if the contract is assigned to a third party?

In [23]:
print_positive_examples("Anti-Assignment")

1. Consequently either this Agreement or any of the respective rights or obligations of the Parties hereunder may be assigned or otherwise transferred, in whole or in part, in any form whatsoever (including by way of change of Control), by either Party subject to the prior written consent of the other Party, which consent shall not be unreasonably withheld or delayed, and any attempt to do so without such consent shall be null and void.

2. Either Party may, without consent of the other Party, assign this Agreement in whole to (i) in the case of RevMed, its successor in interest or assignee or purchaser, as applicable, in the case of a Change of Control or (ii) in the case of Sanofi, its successor in interest or assignee or purchaser, as applicable, in connection with the sale of all or substantially all of its assets to which this Agreement relates, or in connection with a merger, acquisition or similar transaction.

3. Neither this Agreement nor any rights or obligations hereunder sh

In [24]:
print_negative_examples("Anti-Assignment")

1. The construction, interpretation and performance of this Agreement and all transactions under it shall be governed by the law of the State of Israel, without giving effect to choice of law rules, and both Parties consent to jurisdiction by the courts of the City of Haifa.

2. This Agreement shall be governed by and construed in accordance with the laws of the Province of British Columbia and the federal laws of Canada applicable therein, excluding its conflict-of-laws rules.

3. This Agreement shall be governed by and construed under the laws of Hong Kong without regard to choice of laws principles.

4. The Company will have the ability to terminate this Agreement by giving 60 days' prior  written notice to the Contractor.

5. Any books of accounts or records shall not be inspected more than once.



In [25]:
anti_assignment_scorecard = Scorecard(
    category="Anti-Assignment",
    anchor_groups=[
        EvidenceGroup(
            group_id="AA-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="AA-R-001",
                    phrases=[
                        "may be assigned or otherwise transferred",
                        "shall be assignable by a Party without",
                        "may not be assigned by",
                    ],
                    source="annotated training Span",
                    rationale="Directly addresses whether the agreement or contractual rights and obligations may be assigned or transferred.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="AA-A-002",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="AA-R-002",
                    phrases=[
                        "subject to the prior written consent of the other Party",
                        "without the prior written consent of",
                    ],
                    source="annotated training Span",
                    rationale="Identifies a prior-consent requirement that restricts assignment or transfer.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="AA-A-003",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="AA-R-003",
                    phrases=[
                        "shall not sub-license, transfer or otherwise deal with the rights of use",
                    ],
                    source="annotated training Span",
                    rationale="Restricts transfer or other dealing with contractual rights granted under the agreement.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[],
    counterevidence_groups=[],
    veto_groups=[],
)

The positive examples consistently address assignment, transfer, or other dealing with an agreement or contractual rights and obligations, including provisions requiring prior consent and provisions establishing permitted transfers or exceptions. The negative examples concern governing law, termination, and inspection rights and do not establish assignment-related evidence. The scorecard therefore requires an identifiable assignment or transfer context rather than relying on generic terms such as “consent,” “notice,” or “transfer” alone.

## Audit Rights
Description: Does a party have the right to  audit the books, records, or physical locations of the counterparty to ensure compliance with the contract?

In [26]:
print_positive_examples("Audit Rights")

1. The Auditing Party may cause the Auditor to perform such an audit not more than once in any 12-month period, unless a prior audit within the past two years revealed that the amount owed by the Audited Party to the Auditing Party was underpaid in excess of 8% of the amount owed, in which case an audit may be performed no more frequently than twice in any 12-month period.

2. Franchisee shall fully cooperate with Pretzel Time's representatives and independent accountants hired by Pretzel Time to conduct any such inspection or audit.

3. Premier, shall have the right, directly or through its representative, to inspect, copy, and audit all such records upon reasonable request and during normal business hours, acknowledging that access to accounting and purchasing records will be limited to those supporting pass-through materials costs and purchases of Premier specified equipment if any.

4. SHPS shall have the right, upon reasonable prior written notice, to examine, copy and audit such 

In [27]:
print_negative_examples("Audit Rights")

1. In consideration of the License granted and other good and valuable consideration provided by T&B to LEA, LEA shall pay to T&B a base royalty ("Base Royalty") in the amount of [●%] of LEA's monthly Cash Sales for Cash Sales of up to [$●].

2. The revenue share for the Program is stated in Exhibit B.

3. License Renewal and Maintenance Fees will be split evenIy with 50% distributed to Developer and 50% distributed to Distributor, also subject to paragraph 4 of this section 2.3.

4. As part of the exclusive distribution right granted in this Section 2, Vendor hereby grants Distributor the non- exclusive, non-transferable right to use and display Vendor's trademarks, logos, Product photographs and images, Product advertising and promotional copy, including but not limited to the materials contained in Vendor's website, in connection with the promotion, advertising and distribution of the Products.

5. A majority of the Restaurant owners in the Regional Fund may vote to increase the amo

In [28]:
audit_rights_scorecard = Scorecard(
    category="Audit Rights",
    anchor_groups=[
        EvidenceGroup(
            group_id="AR-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="AR-R-001",
                    phrases=[
                        "perform such an audit",
                        "conduct any such inspection or audit",
                        "inspect, copy, and audit all such records",
                        "examine, copy and audit such records",
                    ],
                    source="annotated training Span",
                    rationale="Directly establishes a contractual right to conduct an audit or inspect records.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="AR-A-002",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="AR-R-002",
                    phrases=[
                        "right, directly or through its representative, to inspect, copy, and audit",
                        "have the right, upon reasonable prior written notice, to examine, copy and audit",
                        "full access to all properties, books of account, and records",
                    ],
                    source="annotated training Span",
                    rationale="Establishes an express inspection or record-access right that supports auditing.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[
        EvidenceGroup(
            group_id="AR-S-001",
            effect=EvidenceEffect.SUPPORT,
            rules=[
                Rule(
                    rule_id="AR-R-003",
                    phrases=[
                        "upon reasonable prior written notice",
                        "during normal business hours",
                        "not more than once in any 12-month period",
                    ],
                    source="annotated training Span",
                    rationale="Provides procedural conditions commonly accompanying an express audit or record-inspection right.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
    ],
    counterevidence_groups=[],
    veto_groups=[],
)

The positive examples consistently establish a contractual right to audit, inspect, examine, copy, or access records, with some provisions also specifying notice, frequency, location, business hours, or auditor requirements. The negative examples concern royalties, revenue sharing, license rights, and financial contributions without establishing an audit right. The scorecard therefore treats procedural terms such as notice or frequency as supporting evidence only when connected to an audit or record-inspection context, rather than as independent audit indicators.

## License Grant
Description: Does the contract contain a license granted by one party to its counterparty?

In [29]:
print_positive_examples("License Grant")

1. Subject to the terms and conditions hereof, drkoop.com hereby represents that it has the power and authority to grant, and does hereby grant to Sponsor a non-exclusive, non-transferable, royalty-free, worldwide license to reproduce and display all logos, trademarks, trade names and similar identifying material relating to drkoop.com and, solely as allowed pursuant to this Agreement, to the Dr. C. Everett Koop name (collectively, the "drkoop.com Marks") solely in connection with the promotion, marketing and distribution of the parties and the Sites in accordance with the terms hereof, provided, however, that Sponsor shall, other than as specifically provided for in Section 4.4 of this Agreement, not make any specific use of any drkoop.com Marks without first submitting a sample of such use to drkoop.com and obtaining its prior consent, which consent shall not be unreasonably withheld.

2. During the term of this Agreement, and subject to the terms and conditions hereof, STAAR hereby 

In [30]:
print_negative_examples("License Grant")

1. MediWound shall have an exclusive license under patents and other intellectual property, to develop, use, manufacture, market and sell the Product for burn treatment in humans;

2. Subject to the provisions of this Agreement, Bioeq hereby grants to Licensee an exclusive (even as to Bioeq), milestone- and royalty-bearing, non-transferable license (including the right to grant sublicenses only to the extent permitted by Section 2.1.2) under the Licensed Technology (including the Licensed Patents) to use, sell, have sold, import, have imported or otherwise Commercialize the Licensed Products in the Field in the Territory.

3. Based on the foregoing, and subject to the terms and conditions contained in this Agreement, the Agents severally and not jointly agree to act as, and the Corporation appoints the Agents as, the exclusive agents of the Corporation to offer the Offered Shares for sale on the Closing Date (as defined herein) in the Selling Jurisdictions (as defined herein) on a priv

In [31]:
license_grant_scorecard = Scorecard(
    category="License Grant",
    anchor_groups=[
        EvidenceGroup(
            group_id="LG-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="LG-R-001",
                    phrases=[
                        "does hereby grant to Sponsor a non-exclusive, non-transferable, royalty-free, worldwide license",
                        "grants to Distributor, and Distributor hereby accepts, the limited, nontransferable, nonexclusive right and license to use",
                        "grants Distributor the non-exclusive, non-transferable right to use and display",
                    ],
                    source="annotated training Span",
                    rationale="Directly establishes a contractual grant of a license or right to use specified intellectual property or other identified materials.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="LG-A-002",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="LG-R-002",
                    phrases=[
                        "grants to Pretzel Time and its Affiliates a perpetual and worldwide right to use",
                        "right to use and display Vendor's trademarks, logos",
                    ],
                    source="annotated training Span",
                    rationale="Expressly grants a defined right to use identified trademarks, materials, or business-related intellectual property.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[
        EvidenceGroup(
            group_id="LG-S-001",
            effect=EvidenceEffect.SUPPORT,
            rules=[
                Rule(
                    rule_id="LG-R-003",
                    phrases=[
                        "non-exclusive, non-transferable",
                        "right to grant sublicenses",
                        "worldwide license",
                        "royalty-free",
                    ],
                    source="annotated training Span",
                    rationale="Describes the scope or conditions of an expressly granted license and reinforces a license-grant provision.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
    ],
    counterevidence_groups=[],
    veto_groups=[],
)

The positive examples consistently establish a grant of a right or license to use identified intellectual property, trademarks, materials, or business-related know-how. The negative examples include explicit patent and technology license grants, demonstrating that the presence of the word “license” alone is insufficient to distinguish this category. The scorecard therefore relies on the specific license-grant patterns represented by the positive examples and requires the surrounding licensed subject and contractual context rather than treating generic licensing terminology as sufficient evidence.

## Exclusivity
Description: Is there an exclusive dealing  commitment with the counterparty? This includes a commitment to procure all “requirements” from one party of certain technology, goods, or services or a prohibition on licensing or selling technology, goods or services to third parties, or a prohibition on  collaborating or working with other parties), whether during the contract or  after the contract ends (or both).

In [32]:
print_positive_examples("Exclusivity")

1. Reseller shall not obtain the           TouchStar Software or Support Services (or any software or services           which compete with the TouchStar Software) for sale from any Entity           other than TouchStar or its authorized agents.

2. Nantz Communications and Nantz expressly agree that the Endorsement will not be granted to anyone other than the Company for use during the Term in connection with the advertisement and promotion of sportswear apparel, hats and shoes.

3. During the Term of this Agreement, except as otherwise permitted by this Section 3(a)(v), VS agrees that it shall not enter into the same or substantially similar Commitments with any other company or entity which performs clinical research services the same or similar to those provided by PPD or any PPD affiliate (collectively, "PPD Competitor"), nor shall VS provide preferred pricing to a PPD Competitor which is better than that provided by VS hereunder to PPD.

4. The Company hereby appoints and grants 

In [33]:
print_negative_examples("Exclusivity")

1. Without limiting the foregoing, Magenta may use and disclose Bachem Intellectual Property to the extent necessary in connection with the prosecution, maintenance and enforcement of Magenta Developed Intellectual Property.

2. Subject to the terms and conditions of this Agreement, Licensor hereby grants to Licensee, and Licensee hereby accepts from Licensor, an exclusive, non-transferable (except as set forth in Section 10.7) and non-sublicensable (except as provided in Section 2.1(c)) license to use the Licensed Content in connection with websites associated with the Licensed Domain Names until the earlier of (i) termination or expiration of this Agreement, or (ii) termination or expiration of the Agency Agreement, provided, however, that in the event the Agency Agreement is amended or restated, such amendment or restatement shall not be deemed a termination or expiration of the Agency Agreement.

3. shall retain those licenses granted to it under this Agreement, provided that the l

In [34]:
exclusivity_scorecard = Scorecard(
    category="Exclusivity",
    anchor_groups=[
        EvidenceGroup(
            group_id="EX-A-001",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="EX-R-001",
                    phrases=[
                        "shall not obtain",
                        "will not be granted to anyone other than",
                        "shall not enter into the same or substantially similar Commitments with any other company or entity",
                    ],
                    source="annotated training Span",
                    rationale="Restricts a party from obtaining competing products or services, granting the relevant right to another party, or entering substantially similar arrangements with another entity.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
        EvidenceGroup(
            group_id="EX-A-002",
            effect=EvidenceEffect.ANCHOR,
            rules=[
                Rule(
                    rule_id="EX-R-002",
                    phrases=[
                        "grants Distributor the exclusive right to sell",
                        "exclusivity rights granted to LEA",
                    ],
                    source="annotated training Span",
                    rationale="Expressly establishes an exclusive commercial right or identifies rights granted on an exclusivity basis.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
    ],
    supporting_groups=[
        EvidenceGroup(
            group_id="EX-S-001",
            effect=EvidenceEffect.SUPPORT,
            rules=[
                Rule(
                    rule_id="EX-R-003",
                    phrases=[
                        "PPD Competitor",
                        "exclusive right",
                        "exclusivity rights",
                    ],
                    source="annotated training Span",
                    rationale="Identifies competitor restrictions or expressly describes the commercial right as exclusive, reinforcing an exclusivity provision when accompanied by the relevant contractual context.",
                    reviewer="<reviewer>",
                    behavior="include",
                ),
            ],
        ),
    ],
    counterevidence_groups=[
        EvidenceGroup(
            group_id="EX-C-001",
            effect=EvidenceEffect.COUNTER,
            rules=[
                Rule(
                    rule_id="EX-R-004",
                    phrases=[
                        "exclusive, non-transferable",
                        "exclusive, non-transferable and non-sublicensable",
                    ],
                    source="annotated training Span",
                    rationale="Describes the exclusivity of a license grant rather than establishing a standalone exclusive dealing, appointment, or competitor restriction.",
                    reviewer="<reviewer>",
                    behavior="exclude",
                ),
            ],
        ),
    ],
    veto_groups=[],
)

The positive examples establish exclusive dealing, exclusive appointment or distribution rights, restrictions on comparable arrangements with competitors, or other contractual exclusivity obligations. The negative examples demonstrate that “exclusive” may instead describe the scope of a license grant, while other examples concern retained licenses, inspection rights, or renewal. The scorecard therefore requires exclusivity to operate as a commercial restriction, exclusive right, or comparable contractual exclusivity arrangement rather than treating the word “exclusive” alone as sufficient evidence.